# Spark SQL and Temporary Views

Welcome to the third notebook in our series! After mastering transformations, actions, and how Spark thinks lazily, we will now look at **Spark SQL**.

---

## 1. What is Spark SQL?

Spark SQL is a Spark module for structured data processing. Unlike the basic RDD API, Spark SQL provides Spark with more information about the structure of both the data and the computation being performed.

### Why Use Spark SQL?
* **Unified Interface:** Allows data engineers and data analysts to query data using standard SQL syntax alongside programmatic PySpark APIs.
* **Catalyst Optimizer:** Spark SQL queries go through the Catalyst Optimizer, which applies advanced rule-based and cost-based optimizations (like predicate pushdown and column pruning) to make your queries lightning fast automatically.
* **Data Source Integration:** Seamlessly reads and writes data from diverse formats including JSON, Parquet, Hive, JDBC, and ORC.

## 2. Understanding Spark Views and Tables

To run SQL queries against a DataFrame, Spark requires you to register it as a **View** or a **Table**.

### A. Local Temporary Views (`createOrReplaceTempView`)
* Tied to a single `SparkSession` (i.e., lifespan matches the session).
* Visible only within the notebook/application session where it was created.
* Automatically disappears when the session terminates.

### B. Global Temporary Views (`createOrReplaceGlobalTempView`)
* Tied to the entire Spark application (cross-session within the same app).
* Must be referenced using the `global_temp.` database prefix (e.g., `SELECT * FROM global_temp.my_view`).

### C. Permanent Tables (`saveAsTable`)
* Stored permanently in a metastore (like Hive Metastore).
* Persists data to disk and survives across cluster restarts and distinct application sessions.

## 3. Setting up the Environment

In [ ]:
from pyspark.sql import SparkSession

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("SparkSQLAndViews") \
    .getOrCreate()

# Create sample data: Employee and Department records
employee_data = [
    (1, "Alice", "Engineering", 90000),
    (2, "Bob", "Marketing", 65000),
    (3, "Charlie", "Engineering", 95000),
    (4, "Diana", "HR", 60000),
    (5, "Evan", "Marketing", 72000)
]

columns = ["EmpID", "Name", "Department", "Salary"]
df = spark.createDataFrame(employee_data, columns)

# Display raw DataFrame
df.show()

## 4. Registering and Querying Local Temporary Views

Let's register our DataFrame as a temporary view called `employees` and run native SQL statements on it using `spark.sql()`.

In [ ]:
# Register DataFrame as a local temporary view
df.createOrReplaceTempView("employees")

# Run an ANSI SQL query to find average salary per department
query = """
    SELECT Department, COUNT(*) as Headcount, ROUND(AVG(Salary), 2) as Avg_Salary
    FROM employees
    GROUP BY Department
    ORDER BY Avg_Salary DESC
"""

result_df = spark.sql(query)
result_df.show()

## 5. Working with Global Temporary Views

If you need a view to be accessible across multiple separate Spark sessions within the same application, use a global temporary view.

In [ ]:
# Register as a global temporary view
df.createOrReplaceGlobalTempView("global_employees")

# Querying a global temp view requires the 'global_temp.' prefix
global_query = """
    SELECT Name, Salary 
    FROM global_temp.global_employees 
    WHERE Salary > 70000
"""

spark.sql(global_query).show()

## Summary

In this notebook, we learned:
1. How **Spark SQL** leverages the Catalyst Optimizer for high-performance structured processing.
2. The difference between local temporary views, global temporary views, and permanent tables.
3. How to seamlessly switch between PySpark DataFrame operations and standard SQL strings using `spark.sql()`.